In [210]:
import pandas as pd
import matplotlib.pyplot as plt

1. Data Import & Exploration / Datenimport & Exploration / Importation & Exploration des Données

FR : Le jeu de données provient de la plateforme Open Power System Data (OPSD), qui fournit des données énergétiques à l'échelle européenne. Il contient des séries temporelles horaires sur la consommation (charge) et la production d'énergies renouvelables.

EN : The dataset is sourced from the Open Power System Data (OPSD) platform, providing European-wide energy data. It contains hourly time series for load (consumption) and renewable energy generation.

DE : Der Datensatz stammt von der Plattform Open Power System Data (OPSD), die europaweite Energiedaten bereitstellt. Er enthält stündliche Zeitreihen zur Last (Verbrauch) und zur Erzeugung erneuerbarer Energien.

In [211]:
df = pd.read_csv(r'Data\timeseries60min.csv', parse_dates=['utc_timestamp'],
                 index_col=['utc_timestamp'])
df.head()
df.describe()
print(df.shape)
df.info()

(50401, 299)
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 50401 entries, 2014-12-31 23:00:00+00:00 to 2020-09-30 23:00:00+00:00
Columns: 299 entries, cet_cest_timestamp to UA_load_forecast_entsoe_transparency
dtypes: float64(298), object(1)
memory usage: 115.4+ MB


FR : Nous filtrons le tableau pour ne conserver que les colonnes relatives à l'Allemagne. Toutes les données concernant les autres pays européens sont supprimées pour isoler notre zone d'étude.

EN : We filter the table to keep only the columns related to Germany. All data regarding other European countries is removed to isolate our study area.

DE : Wir filtern die Tabelle, um nur die Spalten mit Bezug zu Deutschland zu behalten. Alle Daten zu anderen europäischen Ländern werden entfernt, um unser Untersuchungsgebiet zu isolieren.

In [212]:
Deutsch_columns = [col for col in df.columns if col.startswith('DE_')]
df_Deutschland = df[Deutsch_columns].copy()
df_Deutschland.head()
print(df_Deutschland.shape)



(50401, 41)


FR : Nous affinons le filtrage en supprimant les colonnes d'interconnexion. Ces colonnes représentent les échanges énergétiques entre l'Allemagne et ses voisins (ex: DE_AT, DE_LU). Nous ne conservons que les données de production et de consommation propres au territoire national.

EN : We refine the filtering by removing interconnection columns. These columns represent energy exchanges between Germany and its neighbors (e.g., DE_AT, DE_LU). We only keep production and consumption data specific to the national territory.

DE : Wir verfeinern die Filterung, indem wir die Interkonnektionsspalten entfernen. Diese Spalten stellen den Energieaustausch zwischen Deutschland und seinen Nachbarn dar (z. B. DE_AT, DE_LU). Wir behalten nur die produktions- und verbrauchsbezogenen Daten für das Staatsgebiet.

In [213]:

#df_Deutschland.dropna(axis='index', how='all')
#print (df_Deutschland.shape)
#df_Deutschland.isnull().head()
#df_Deutschland.count()
df_columnswithoutLU = [col for col in df_Deutschland if '_LU_'not in col]
df_Deutschland = df_Deutschland[df_columnswithoutLU].copy()
#df_Deutschland.count()
#df_Deutschland.info()
df_Deutschland.shape
#df_Deutschland.head()







(50401, 34)

FR :  Traitement des valeurs manquantes et Filtrage des lignes
Nous analysons la présence de données nulles (NaN), particulièrement pour le profil solaire. Nous choisissons de supprimer les lignes où la consommation réelle (DE_load_actual) est manquante, car cette variable est indispensable à notre analyse comparative.

EN :  Handling Missing Values and Row Filtering
We analyze the presence of null data (NaN), specifically for the solar profile. We choose to drop rows where the actual load (DE_load_actual) is missing, as this variable is essential for our comparative analysis.

DE :  Umgang mit fehlenden Werten und Zeilenfilterung
Wir analysieren das Vorhandensein von Nullwerten (NaN), insbesondere für das Solarprofil. Wir entscheiden uns, Zeilen zu löschen, in denen die tatsächliche Last (DE_load_actual) fehlt, da diese Variable für unsere vergleichende Analyse unerlässlich ist.

In [214]:
df_Deutschland['DE_solar_profile'].isna()
print(df_Deutschland.iloc[0])
df_Deutschland = df_Deutschland.dropna(subset=['DE_load_actual_entsoe_transparency'])
df_Deutschland.head()


DE_load_actual_entsoe_transparency                     NaN
DE_load_forecast_entsoe_transparency                   NaN
DE_solar_capacity                                  37248.0
DE_solar_generation_actual                             NaN
DE_solar_profile                                       NaN
DE_wind_capacity                                   27913.0
DE_wind_generation_actual                              NaN
DE_wind_profile                                        NaN
DE_wind_offshore_capacity                            667.0
DE_wind_offshore_generation_actual                     NaN
DE_wind_offshore_profile                               NaN
DE_wind_onshore_capacity                           27246.0
DE_wind_onshore_generation_actual                      NaN
DE_wind_onshore_profile                                NaN
DE_50hertz_load_actual_entsoe_transparency             NaN
DE_50hertz_load_forecast_entsoe_transparency           NaN
DE_50hertz_solar_generation_actual                     N

,DE_load_actual_entsoe_transparency,DE_load_forecast_entsoe_transparency,DE_solar_capacity,DE_solar_generation_actual,DE_solar_profile,DE_wind_capacity,DE_wind_generation_actual,DE_wind_profile,DE_wind_offshore_capacity,DE_wind_offshore_generation_actual,...,DE_tennet_load_actual_entsoe_transparency,DE_tennet_load_forecast_entsoe_transparency,DE_tennet_solar_generation_actual,DE_tennet_wind_generation_actual,DE_tennet_wind_offshore_generation_actual,DE_tennet_wind_onshore_generation_actual,DE_transnetbw_load_actual_entsoe_transparency,DE_transnetbw_load_forecast_entsoe_transparency,DE_transnetbw_solar_generation_actual,DE_transnetbw_wind_onshore_generation_actual
utc_timestamp,,,,,,,,,,,,,,,,,,,,,
2015-01-01 00:00:00+00:00,41151.0,39723.0,37248.0,NaN,NaN,27913.0,8852.0,0.3171,667.0,517.0,...,13841.0,13362.0,NaN,3866.0,469.0,3398.0,5307.0,4703.0,NaN,5.0
2015-01-01 01:00:00+00:00,40135.0,38813.0,37248.0,NaN,NaN,27913.0,9054.0,0.3244,667.0,514.0,...,13267.0,12858.0,NaN,3974.0,466.0,3508.0,5087.0,4562.0,NaN,7.0
2015-01-01 02:00:00+00:00,39106.0,38490.0,37248.0,NaN,NaN,27913.0,9070.0,0.3249,667.0,518.0,...,12702.0,12611.0,NaN,4194.0,470.0,3724.0,4906.0,4517.0,NaN,8.0
2015-01-01 03:00:00+00:00,38765.0,38644.0,37248.0,NaN,NaN,27913.0,9163.0,0.3283,667.0,520.0,...,12452.0,12490.0,NaN,4446.0,473.0,3973.0,4865.0,4601.0,NaN,11.0
2015-01-01 04:00:00+00:00,38941.0,38773.0,37248.0,NaN,NaN,27913.0,9231.0,0.3307,667.0,520.0,...,12454.0,12464.0,NaN,4671.0,474.0,4198.0,4685.0,4519.0,NaN,6.0


FR : Exclusion des données par gestionnaire de réseau (TSO)
Nous filtrons le dataset pour supprimer les colonnes spécifiques aux quatre grands gestionnaires de réseau allemands (50Hertz, TenneT, Amprion, TransnetBW). Cette étape permet de ne conserver que les agrégats nationaux et d'éviter les redondances dans l'analyse.

EN :  Exclusion of Transmission System Operator (TSO) Data
We filter the dataset to remove columns specific to the four major German grid operators (50Hertz, TenneT, Amprion, TransnetBW). This step ensures that only national aggregates are kept, avoiding redundancies in the analysis.

DE :  Ausschluss der Übertragungsnetzbetreiber-Daten (ÜNB)
Wir filtern den Datensatz, um die spezifischen Spalten der vier großen deutschen Übertragungsnetzbetreiber (50Hertz, TenneT, Amprion, TransnetBW) zu entfernen. Dieser Schritt dient dazu, nur die nationalen Aggregate zu behalten und Redundanzen in der Analyse zu vermeiden.


In [215]:

company_name = ['50hertz', 'tennet', 'amprion', 'transnetbw']
KeepColumns = [col for col in df_Deutschland.columns 
               if not any(name in col.lower() for name in company_name)]
df_Deutschland = df_Deutschland[KeepColumns].copy()
print(df_Deutschland.columns.tolist())
df_Deutschland.shape
df_Deutschland.describe()


['DE_load_actual_entsoe_transparency', 'DE_load_forecast_entsoe_transparency', 'DE_solar_capacity', 'DE_solar_generation_actual', 'DE_solar_profile', 'DE_wind_capacity', 'DE_wind_generation_actual', 'DE_wind_profile', 'DE_wind_offshore_capacity', 'DE_wind_offshore_generation_actual', 'DE_wind_offshore_profile', 'DE_wind_onshore_capacity', 'DE_wind_onshore_generation_actual', 'DE_wind_onshore_profile']


,DE_load_actual_entsoe_transparency,DE_load_forecast_entsoe_transparency,DE_solar_capacity,DE_solar_generation_actual,DE_solar_profile,DE_wind_capacity,DE_wind_generation_actual,DE_wind_profile,DE_wind_offshore_capacity,DE_wind_offshore_generation_actual,DE_wind_offshore_profile,DE_wind_onshore_capacity,DE_wind_onshore_generation_actual,DE_wind_onshore_profile
count,50400.000000,50376.000000,43799.000000,50297.000000,43696.000000,43799.000000,50326.000000,43725.000000,43799.000000,50326.000000,43725.000000,43799.000000,50328.000000,43727.000000
mean,55492.468552,54791.384231,42378.132240,4566.042905,0.101902,39972.882098,11552.234650,0.278908,3261.083312,1970.480984,0.570559,36711.810840,9581.469321,0.254253
std,10015.431042,9496.890313,4306.371168,6940.267590,0.155718,7262.640092,9076.350769,0.211566,1359.147670,1567.541202,0.393074,5961.336813,7987.848026,0.206999
min,31307.000000,28824.000000,37248.000000,0.000000,0.000000,27913.000000,135.000000,0.003800,667.000000,0.000000,0.000000,27246.000000,119.000000,0.004100
25%,47106.000000,46987.000000,38810.000000,0.000000,0.000000,33737.000000,4506.000000,0.114900,2219.000000,586.000000,0.207000,31519.000000,3534.000000,0.097400
50%,55092.000000,54731.500000,40941.000000,173.000000,0.003000,39808.000000,9015.000000,0.221100,3115.000000,1640.000000,0.538800,36693.000000,7128.500000,0.192000
75%,64309.250000,62877.250000,46092.000000,7342.000000,0.162300,47730.000000,16113.750000,0.389300,4486.000000,3066.000000,0.899300,43243.000000,13302.250000,0.351700
max,77549.000000,75912.000000,50508.000000,32947.000000,0.687300,50452.000000,46064.000000,1.078000,5742.000000,6901.000000,1.497900,44710.000000,40752.000000,1.088300


FR :  Analyse de la complétude des données de capacité
Nous identifions précisément la dernière occurrence de données valides pour la capacité solaire. Cette étape permet de confirmer jusqu'à quelle date les métadonnées techniques sont disponibles avant de procéder à une éventuelle extrapolation ou un remplissage des valeurs manquantes.

EN :  Capacity Data Completeness Analysis
We precisely identify the last occurrence of valid data for solar capacity. This step confirms up to which date technical metadata is available before proceeding with any potential extrapolation or filling of missing values.

DE :  Analyse der Vollständigkeit der Kapazitätsdaten
Wir identifizieren präzise das letzte Vorkommen gültiger Daten für die Solarkapazität. Dieser Schritt bestätigt, bis zu welchem Datum technische Metadaten verfügbar sind, bevor eine mögliche Extrapolation oder das Auffüllen fehlender Werte erfolgt.

In [216]:


index_valid = df_Deutschland['DE_solar_capacity'].last_valid_index()
valeur_final = df_Deutschland.loc[index_valid, 'DE_solar_capacity']
print (f"valeur trouve le {index_valid} : {valeur_final}")
LastDateCapa = df_Deutschland.dropna(subset = ['DE_solar_capacity'])
Truedate = LastDateCapa.index[-1]
print(f"{Truedate}")
df_Deutschland.describe()




valeur trouve le 2019-12-30 22:00:00+00:00 : 50508.0
2019-12-30 22:00:00+00:00


,DE_load_actual_entsoe_transparency,DE_load_forecast_entsoe_transparency,DE_solar_capacity,DE_solar_generation_actual,DE_solar_profile,DE_wind_capacity,DE_wind_generation_actual,DE_wind_profile,DE_wind_offshore_capacity,DE_wind_offshore_generation_actual,DE_wind_offshore_profile,DE_wind_onshore_capacity,DE_wind_onshore_generation_actual,DE_wind_onshore_profile
count,50400.000000,50376.000000,43799.000000,50297.000000,43696.000000,43799.000000,50326.000000,43725.000000,43799.000000,50326.000000,43725.000000,43799.000000,50328.000000,43727.000000
mean,55492.468552,54791.384231,42378.132240,4566.042905,0.101902,39972.882098,11552.234650,0.278908,3261.083312,1970.480984,0.570559,36711.810840,9581.469321,0.254253
std,10015.431042,9496.890313,4306.371168,6940.267590,0.155718,7262.640092,9076.350769,0.211566,1359.147670,1567.541202,0.393074,5961.336813,7987.848026,0.206999
min,31307.000000,28824.000000,37248.000000,0.000000,0.000000,27913.000000,135.000000,0.003800,667.000000,0.000000,0.000000,27246.000000,119.000000,0.004100
25%,47106.000000,46987.000000,38810.000000,0.000000,0.000000,33737.000000,4506.000000,0.114900,2219.000000,586.000000,0.207000,31519.000000,3534.000000,0.097400
50%,55092.000000,54731.500000,40941.000000,173.000000,0.003000,39808.000000,9015.000000,0.221100,3115.000000,1640.000000,0.538800,36693.000000,7128.500000,0.192000
75%,64309.250000,62877.250000,46092.000000,7342.000000,0.162300,47730.000000,16113.750000,0.389300,4486.000000,3066.000000,0.899300,43243.000000,13302.250000,0.351700
max,77549.000000,75912.000000,50508.000000,32947.000000,0.687300,50452.000000,46064.000000,1.078000,5742.000000,6901.000000,1.497900,44710.000000,40752.000000,1.088300


FR :  Imputation des données de capacité (Forward Fill)
Les données de capacité solaire sont structurellement croissantes et mises à jour moins fréquemment que la production. Après le 30/12/2019 à 22h, les données sont manquantes. Nous avons décidé de remplir ces vides en prolongeant la dernière valeur connue (50 508.02 MW), car la capacité installée ne peut pas régresser d'un jour à l'autre.

EN :  Capacity Data Imputation (Forward Fill)
Solar capacity data is structurally increasing and updated less frequently than production. After 12/30/2019 at 10 PM, data is missing. We decided to fill these gaps by extending the last known value (50,508.02 MW), as installed capacity cannot decrease from one day to the next.

DE :  Imputation der Kapazitätsdaten (Forward Fill)
Die Solarkapazitätsdaten sind strukturell steigend und werden seltener aktualisiert als die Erzeugung. Nach dem 30.12.2019 um 22:00 Uhr fehlen die Daten. Wir haben uns entschieden, diese Lücken durch Verlängerung des zuletzt bekannten Wertes (50.508,02 MW) zu füllen, da die installierte Kapazität von einem Tag auf den anderen nicht sinken kann.

In [217]:
df_Deutschland ['DE_solar_capacity'] = df_Deutschland ['DE_solar_capacity'].ffill()
df_Deutschland ['DE_solar_capacity'].tail()
df_Deutschland.describe()
df_Deutschland.head()
df_Deutschland.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 50400 entries, 2015-01-01 00:00:00+00:00 to 2020-09-30 23:00:00+00:00
Data columns (total 14 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   DE_load_actual_entsoe_transparency    50400 non-null  float64
 1   DE_load_forecast_entsoe_transparency  50376 non-null  float64
 2   DE_solar_capacity                     50400 non-null  float64
 3   DE_solar_generation_actual            50297 non-null  float64
 4   DE_solar_profile                      43696 non-null  float64
 5   DE_wind_capacity                      43799 non-null  float64
 6   DE_wind_generation_actual             50326 non-null  float64
 7   DE_wind_profile                       43725 non-null  float64
 8   DE_wind_offshore_capacity             43799 non-null  float64
 9   DE_wind_offshore_generation_actual    50326 non-null  float64
 10  DE_wind_offshore_profile           

FR :  Traitement des lacunes de production réelle
Nous avons identifié des données manquantes dans DE_solar_generation_actual. Contrairement à la capacité, ces valeurs sont volatiles. Nous appliquons une interpolation pour les micro-coupures et vérifions la cohérence avec les données de prévisions (forecast) pour garantir la continuité des séries temporelles.

EN :  Handling Actual Generation Gaps
We identified missing data in DE_solar_generation_actual. Unlike capacity, these values are volatile. We apply interpolation for micro-outages and cross-reference with forecast data (forecast) to ensure the continuity of the time series.

DE :  Umgang mit Lücken in der tatsächlichen Erzeugung
Wir haben fehlende Daten in DE_solar_generation_actual identifiziert. Im Gegensatz zur Kapazität sind diese Werte volatil. Wir wenden eine Interpolation für Mikrounterbrechungen an und prüfen die Konsistenz mit den Prognosedaten (forecast), um die Kontinuität der Zeitreihen zu gewährleisten.

In [218]:
df_Deutschland['DE_solar_generation_actual']

utc_timestamp
2015-01-01 00:00:00+00:00    NaN
2015-01-01 01:00:00+00:00    NaN
2015-01-01 02:00:00+00:00    NaN
2015-01-01 03:00:00+00:00    NaN
2015-01-01 04:00:00+00:00    NaN
                            ... 
2020-09-30 19:00:00+00:00    0.0
2020-09-30 20:00:00+00:00    0.0
2020-09-30 21:00:00+00:00    0.0
2020-09-30 22:00:00+00:00    0.0
2020-09-30 23:00:00+00:00    0.0
Name: DE_solar_generation_actual, Length: 50400, dtype: float64